In [1]:
import os

# Get the current directory of the notebook
notebook_dir = os.getcwd()


# Build the path to the data
data_path = os.path.join(notebook_dir, "data", "raw", "invoice_export_q1.xlsx")

print("Notebook is here:", notebook_dir)
print("Data is at:", data_path)

Notebook is here: /Users/huien/Documents/opus-tuition/src/data_cleaning
Data is at: /Users/huien/Documents/opus-tuition/src/data_cleaning/data/raw/invoice_export_q1.xlsx


In [2]:
# Install necessary packages
!pip install pandas
!pip install openpyxl
!pip install xlsxwriter


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import openpyxl
import pandas as pd
import re
from datetime import date, datetime
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum

#### Find and add header if it doesn't exist

In [4]:
import openpyxl

def find_highlighted_header(file_path, max_scan_rows=30):
    try:
        # Load the workbook safely
        wb = openpyxl.load_workbook(file_path, data_only=True)
        first_sheet_name = wb.sheetnames[0]
        sheet = wb[first_sheet_name]

        # Loop through up to 30 rows
        for row_index in range(1, max_scan_rows + 1):
            has_bold = False
            has_color = False
            row_values = []

            # Loop through every cell in current row
            for cell in sheet[row_index]:
                cell_value = cell.value

                # Check if there is text in the cell
                if cell_value is not None and cell_value != "":
                    row_values.append(cell_value)

                    # Check if cell's font is bolded
                    if cell.font and getattr(cell.font, 'bold', False):
                        has_bold = True

                    # Check if a fill pattern actually exists and isn't the default "none" string
                    if cell.fill:
                        # Verify the cell has a solid fill pattern
                        is_solid = getattr(cell.fill, 'fill_type', None) == 'solid'

                        # Verify that background fill color exists and has a valid RGB value
                        start_color = getattr(cell.fill, 'start_color', None)
                        has_rgb = start_color and start_color.rgb is not None

                        # If both conditions are met, the cell has a custom background color
                        if is_solid and has_rgb:
                            has_color = True

            # Check if the row have multiple data columns
            has_multiple_columns = len(row_values) > 1
            # Check if each cell in the current row is unique
            is_unique = len(row_values) == len(set(row_values))

            if has_multiple_columns and has_color and has_bold and is_unique:
                print(f"Header successfully matched at row {row_index} ({len(row_values)} unique columns found)!")
                return first_sheet_name, (row_index - 1), True

        # Fallback if the loop finishes without matching any specific criteria
        print("Notice: No highlighted header matched standard criteria. Defaulting to row 0.")
        return first_sheet_name, 0, False

    except FileNotFoundError:
        print(f"Error: The file at path '{file_path}' was not found.")
        return None, 0, False
    except Exception as e:
        print(f"Error reading or processing the Excel file header: {e}")
        return None, 0, False


In [5]:
def is_valid_date(raw_val) -> bool:
    # Define the exact formats expected
    expected_formats = [
        "%Y-%m-%d",   # 2025-03-12
        "%m/%d/%Y",   # 01/05/2025
        "%d-%m-%Y",   # 25-04-2025
        "%d/%m/%y",   # 25/04/26
        "%d %b %Y",   # 12 Feb 2025
        "%B %d, %Y",  # July 7, 2025
        "%d-%b-%Y"    # 15-Oct-2025
    ]

    try:
        # Return False immediately for empty, null, or blank values
        if pd.isna(raw_val) or str(raw_val).strip() in ["", "None", "NaN", "nan"]:
            return False

        clean_val = str(raw_val).strip()

        # Return True the moment a format matches successfully
        for fmt in expected_formats:
            try:
                datetime.strptime(clean_val, fmt)
                return True
            except ValueError:
                continue

    except Exception as e:
        # Catch unexpected type conversion errors or pandas evaluation crashes
        print(f"Error checking date validity for value '{raw_val}': {e}")
        return False

    # Return False if all formatting options fail
    return False

In [6]:
def extract_numeric_value(value) -> int:
    try:
        # Extract the first block of digits, periods, or commas
        match = re.search(r"[\d.,]+", str(value))
        if match:
            # Remove formatting commas
            clean_num = match.group(0).replace(",", "")
            # Convert to float first to handle decimals like "50.00", then truncate to integer
            return int(float(clean_num))
        return 0
    except (ValueError, TypeError):
        return 0


In [7]:
def extract_float_value(value) -> float:
    try:
        # Convert value to string and extract only the first numeric block
        match = re.search(r"[\d.,]+", str(value))
        if match:
            # Strip out commas and convert the clean number to a float
            clean_num = match.group(0).replace(",", "")
            return float(clean_num)
        return 0.0
    except (ValueError, TypeError):
        return 0.0

In [8]:
import pandas as pd

# Look for custom highlighted headers first
sheet_name, rows_to_skip, header_found = find_highlighted_header(data_path)

if header_found:
    df = pd.read_excel(data_path, sheet_name=sheet_name, skiprows=rows_to_skip)
else:
    # Load without headers to start heuristic mapping
    df = pd.read_excel(data_path, sheet_name=sheet_name, header=None)

    max_rows_to_check = min(30, len(df))
    session_notes_words_to_check = ["reading", "covered", "timed"]
    invoice_notes_phases_to_check = ["awaiting bank transfer", "no response from parent"]
    student_names = ["lim wei jie"]
    tutor_names = ["ahmad rizwan"]

    first_data_row_index = None
    existing_headers = []
    column_samples = {}
    pass1_headers = []

    class PaymentStatus(Enum):
        PAID = "paid"
        PENDING = "pending"

    class AccountStatus(Enum):
        ACTIVE = "active"
        INACTIVE = "inactive"
        PENDING = "pending"

    try:
        # ==========================================
        # PASS 1: Identify Structural Anchor Columns
        # ==========================================
        for col_index in range(df.shape[1]):
            column_sample = df[col_index].iloc[:max_rows_to_check].tolist()
            column_samples[col_index] = column_sample
            matched_header = None

            for current_row_index, value in enumerate(column_sample):
                clean_val = str(value).strip().lower() if pd.notna(value) else ""

                if clean_val.startswith("log"):
                    matched_header = "Log ID"
                elif clean_val.startswith("tas"):
                    matched_header = "Assignment ID"
                elif clean_val.startswith("inv"):
                    matched_header = "Invoice ID"
                elif any(clean_val == name for name in student_names):
                    matched_header = "Student Name"
                elif any(clean_val == name for name in tutor_names):
                    matched_header = "Tutor Name"
                elif clean_val in ["1.0", "1.5", "2"]:
                    matched_header = "Duration (Hours)"
                elif clean_val in ["present", "absent", "late"]:
                    matched_header = "Attendance Status"
                elif clean_val in ["english", "mathematics", "science"]:
                    matched_header = "Subject"
                elif "primary" in clean_val or "secondary" in clean_val or "junior college" in clean_val:
                    matched_header = "Level"
                elif clean_val == PaymentStatus.PAID.value:
                    matched_header = "Payment Status"
                elif clean_val in [AccountStatus.ACTIVE.value, AccountStatus.INACTIVE.value]:
                    matched_header = "Status"
                elif clean_val.endswith("@tutors.com"):
                    matched_header = "Contact Email"

                if matched_header:
                    if matched_header not in existing_headers:
                        existing_headers.append(matched_header)
                    if first_data_row_index is None or current_row_index < first_data_row_index:
                        first_data_row_index = current_row_index
                    break

            pass1_headers.append(matched_header)

        # ==========================================
        # PASS 2: Resolve Context-Dependent Columns
        # ==========================================
        new_headers = []
        is_lesson_log = "Log ID" in existing_headers and "Assignment ID" in existing_headers
        is_invoice = "Invoice ID" in existing_headers and "Assignment ID" in existing_headers
        is_assignment = "Assignment ID" in existing_headers and "Level" in existing_headers and "Subject" in existing_headers

        for col_index, matched_header in enumerate(pass1_headers):
            if matched_header is None:
                for current_row_index, value in enumerate(column_samples[col_index]):
                    clean_val = str(value).strip().lower() if pd.notna(value) else ""

                    try:
                        numeric_val = extract_float_value(clean_val)
                    except Exception as e:
                        raise ValueError(f"Failed extracting numeric value from row {current_row_index}, col {col_index} ('{clean_val}'): {e}")

                    if is_lesson_log:
                        if is_valid_date(clean_val):
                            matched_header = "Session Date"
                        elif numeric_val > 45:
                            matched_header = "Fees Charged"
                        elif len(clean_val.split()) >= 2 and any(word in clean_val.split() for word in session_notes_words_to_check):
                            matched_header = "Session Notes"

                    elif is_invoice:
                        if is_valid_date(clean_val):
                            matched_header = "Date"
                        elif numeric_val > 45:
                            matched_header = "Amount"
                        elif any(clean_val == phase for phase in invoice_notes_phases_to_check):
                            matched_header = "Notes"

                    elif is_assignment:
                        if is_valid_date(clean_val):
                            matched_header = "Start Date"
                        elif numeric_val > 45:
                            matched_header = "Hourly Rate"

                    if matched_header:
                        if first_data_row_index is None or current_row_index < first_data_row_index:
                            first_data_row_index = current_row_index
                        break

            # Assign a fallback structural index label if unmatched
            new_headers.append(matched_header if matched_header else f"Column_{col_index+1}")

        if is_invoice:
                date_cols = [i for i, h in enumerate(new_headers) if h == "Date_Candidate"]
                if len(date_cols) >= 2:
                    # Compare dates in the first available data row to determine identity
                    d1 = pd.to_datetime(df.iloc[first_data_row_index, date_cols[0]])
                    d2 = pd.to_datetime(df.iloc[first_data_row_index, date_cols[1]])
                    if d1 < d2:
                        new_headers[date_cols[0]], new_headers[date_cols[1]] = "Invoice Date", "Payment Date"
                    else:
                        new_headers[date_cols[0]], new_headers[date_cols[1]] = "Payment Date", "Invoice Date"
                elif len(date_cols) == 1:
                    new_headers[date_cols[0]] = "Invoice Date"

        if not any(new_headers):
            raise ValueError("Mapping failed: No headers could be identified.")

        df.columns = new_headers

        # Crop data to match discovered table boundaries
        if first_data_row_index is not None:
            df = df.iloc[first_data_row_index:].reset_index(drop=True)

        print("Heuristic column mapping completed successfully!")

    except ValueError as ve:
        print(f"Data Validation Error: {ve}")
    except Exception as e:
        print(f"An unexpected error occurred during column mapping: {e}")

Header successfully matched at row 1 (8 unique columns found)!


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Invoice ID      19 non-null     str  
 1   Tutor ID        19 non-null     str  
 2   Student Name    19 non-null     str  
 3   Invoice Date    19 non-null     str  
 4   Amount          19 non-null     str  
 5   Payment Status  19 non-null     str  
 6   Payment Date    13 non-null     str  
 7   Notes           4 non-null      str  
dtypes: str(8)
memory usage: 1.3 KB


In [10]:
df

,Invoice ID,Tutor ID,Student Name,Invoice Date,Amount,Payment Status,Payment Date,Notes
0,INV-2025-001,TAS-001,Lim Wei Jie,2025-01-31,SGD 165.00,Paid,2025-02-05,NaN
1,INV-2025-002,TAS-002,Tan Xiu Ying,2025-03-31,SGD 90.00,Pending,NaN,Awaiting bank transfer
2,INV-2025-003,TAS-003,Muhammad Hafiz,2025-02-28,SGD 240.00,Paid,2025-03-03,NaN
3,INV-2025-004,TAS-004,Chloe Wong,2025-03-31,SGD 75.00,PENDING,NaN,No response from parent
4,INV-2025-005,TAS-006,Ethan Ng,2025-04-30,SGD 70.00,pend,NaN,NaN
5,INV-2025-006,TAS-007,Hafiz Bin Aris,2025-06-30,SGD 112.50,Pending,NaN,Note trailing space
6,INV-2025-007,TAS-008,Aisha Binte Yusof,2025-07-31,SGD 90.00,Paid,2025-08-02,NaN
7,INV-2025-008,TAS-010,Darren Foo,2025-08-31,SGD 165.00,Paid,2025-09-01,NaN
8,INV-2025-001,TAS-001,Lim Wei Jie,2025-01-31,SGD 165.00,Paid,2025-02-05,NaN
9,INV-2025-009,TAS-011,Raeanne Ong,2025-09-30,SGD 120.00,Paid,2025-10-04,NaN


#### Clean dataset

In [11]:
# Create a brand new copy first
df_copy = df.copy()

# Clean the column names on the new copy
df_copy.columns = df_copy.columns.astype(str).str.strip()

# Strip text data columns on the new copy only
for col in df_copy.select_dtypes(include=["object", "string"]):
    df_copy[col] = df_copy[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df_copy[col] = df_copy[col].replace(["nan", "None", ""], None)

# Drop blank rows where all values are nulls
df_cleaned = df_copy.dropna(how="all")

# Remove duplicates safely
try:
    if "Invoice ID" in df_cleaned.columns:
        df_cleaned = df_cleaned.drop_duplicates(subset=["Invoice ID"], keep="last")

    elif "Attendance ID" in df_cleaned.columns:
        df_cleaned = df_cleaned.drop_duplicates(subset=["Attendance ID"], keep="last")

        # Second, drop based on the combination of details if they exist
        secondary_cols = ["Tutor Name", "Student Name", "Start Date"]
        if all(col in df_cleaned.columns for col in secondary_cols):
            df_cleaned = df_cleaned.drop_duplicates(subset=secondary_cols, keep="last")
        else:
            print("Warning: Secondary columns for 'Attendance ID' duplicate check are missing. Skipping second drop.")

    elif "Log ID" in df_cleaned.columns:
        # First, drop based on Log ID
        df_cleaned = df_cleaned.drop_duplicates(subset=["Log ID"], keep="last")

    else:
        # Fallback keeps the cleaned data if no specific ID column matches
        print("Notice: No matching ID column found. Keeping cleaned dataframe as is.")

except Exception as e:
    print(f"Error during duplicate removal: {e}. Falling back to clean dataframe copy.")
    df_cleaned = df_copy.copy()


In [12]:
def process_string_columns(df, columns_to_convert):
    """
    Checks if specific columns exist and converts them to string dtype safely.
    """
    for column in columns_to_convert:
        try:
            if column in df.columns:
                df[column] = df[column].astype("string")
                print(f"Successfully converted '{column}' to string dtype.")
            else:
                print(f"Skipping: Column '{column}' does not exist.")
        except Exception as e:
            print(f"Error converting column '{column}': {e}")

    return df

# 1. Define your target columns
columns_to_stringify = [
    "Invoice ID",
    "Tutor ID",
    "Student Name",
    "Tutor Name",
    "Payment Status",
    "Notes",
    "Log ID",
    "Assignment ID",
    "Attendance Status",
    "Session Notes",
    "Subject",
    "Level",
    "Status",
    "Contact Email"
]

# 2. Process them all with a single function call
df_cleaned = process_string_columns(df_cleaned, columns_to_stringify)

Successfully converted 'Invoice ID' to string dtype.
Successfully converted 'Tutor ID' to string dtype.
Successfully converted 'Student Name' to string dtype.
Skipping: Column 'Tutor Name' does not exist.
Successfully converted 'Payment Status' to string dtype.
Successfully converted 'Notes' to string dtype.
Skipping: Column 'Log ID' does not exist.
Skipping: Column 'Assignment ID' does not exist.
Skipping: Column 'Attendance Status' does not exist.
Skipping: Column 'Session Notes' does not exist.
Skipping: Column 'Subject' does not exist.
Skipping: Column 'Level' does not exist.
Skipping: Column 'Status' does not exist.
Skipping: Column 'Contact Email' does not exist.


In [13]:
def capitalise_string_columns(df, columns_to_format):
    """
    Safely converts specified string columns to lowercase, then capitalises
    the first letter of each word.
    """
    for column in columns_to_format:
        if column in df.columns:
            try:
                # Python's built-in command MUST use 'z'
                df[column] = df[column].str.lower().str.capitalize()
                print(f"Successfully capitalised strings in column: '{column}'")
            except Exception as e:
                print(f"Error capitalising column '{column}': {e}")
        else:
            print(f"Skipping: Column '{column}' does not exist.")

    return df

# Define the list of columns for cleaning
target_cols = ["Payment Status", "Attendance Status", "Subject", "Status"]

# Apply the function to all target columns at once
df_cleaned = capitalise_string_columns(df_cleaned, target_cols)

Successfully capitalised strings in column: 'Payment Status'
Skipping: Column 'Attendance Status' does not exist.
Skipping: Column 'Subject' does not exist.
Skipping: Column 'Status' does not exist.


In [14]:
def titlecase_columns(df, columns_to_format):
    """
    Capitalises the first letter of every word.
    """
    for column in columns_to_format:
        if column in df.columns:
            try:
                df[column] = df[column].str.title()
                print(f"Successfully applied Title Case to: '{column}'")
            except Exception as e:
                print(f"Error processing title case on '{column}': {e}")
        else:
            print(f"Skipping: Column '{column}' does not exist.")
    return df

# Define the list of columns for cleaning
target_cols = ["Student Name", "Teacher Name", "Level"]

# Apply the function to all target columns at once
df_cleaned = titlecase_columns(df_cleaned, target_cols)

Successfully applied Title Case to: 'Student Name'
Skipping: Column 'Teacher Name' does not exist.
Skipping: Column 'Level' does not exist.


In [15]:
def standardise_column_values(df):
    """
    Applies specific cleanup rules to Payment Status, Attendance Status,
    Duration (Hours), and Fees Charged columns if they exist.
    """
    # 1. Clean up "Payment Status"
    if "Payment Status" in df.columns:
        try:
            # Replace "Pend" with "Pending"
            df["Payment Status"] = df["Payment Status"].replace({"Pend": "Pending"})
        except Exception as e:
            print(f"Error processing 'Payment Status': {e}")
    else:
        print("Skipping 'Payment Status': Column does not exist.")

    # 2. Clean up "Attendance Status"
    if "Attendance Status" in df.columns:
        try:
            df["Attendance Status"] = df["Attendance Status"].fillna("Absent")
        except Exception as e:
            print(f"Error processing 'Attendance Status': {e}")
    else:
        print("Skipping 'Attendance Status': Column does not exist.")

    # 3. Clean up "Duration (Hours)"
    if "Duration (Hours)" in df.columns:
        try:
            # Replace null values to 1.0 by default
            df["Duration (Hours)"] = df["Duration (Hours)"].fillna(1.0)
        except Exception as e:
            print(f"Error processing 'Duration (Hours)': {e}")
    else:
        print("Skipping 'Duration (Hours)': Column does not exist.")

    # 4. Clean up "Fees Charged"
    if "Fees Charged" in df.columns:
        try:
            df["Fees Charged"] = df["Fees Charged"].replace(r'^[A-Za-z]+$', 0, regex=True)
        except Exception as e:
            print(f"Error processing 'Fees Charged': {e}")
    else:
        print("Skipping 'Fees Charged': Column does not exist.")

    if "Subject" in df.columns:
        try:
            df_cleaned["Subject"] = df_cleaned["Subject"].replace({"Maths": "Mathematics"})
        except Exception as e:
            print(f"Error processing 'Subject': {e}")
    else:
        print("Skipping 'Subject': Column does not exist.")

    return df

df_cleaned = standardise_column_values(df_cleaned)

Skipping 'Attendance Status': Column does not exist.
Skipping 'Duration (Hours)': Column does not exist.
Skipping 'Fees Charged': Column does not exist.
Skipping 'Subject': Column does not exist.


In [16]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 13 entries, 3 to 18
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Invoice ID      13 non-null     string
 1   Tutor ID        13 non-null     string
 2   Student Name    13 non-null     string
 3   Invoice Date    13 non-null     str   
 4   Amount          13 non-null     str   
 5   Payment Status  13 non-null     string
 6   Payment Date    8 non-null      str   
 7   Notes           3 non-null      string
dtypes: str(3), string(5)
memory usage: 936.0 bytes


In [17]:
def parse_date(df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    expected_formats = [
        "%Y-%m-%d",     # 2025-03-12
        "%m/%d/%Y",     # 01/05/2025
        "%d-%m-%Y",     # 25-04-2025
        "%d %b %Y",     # 12 Feb 2025
        "%B %d, %Y",    # July 7, 2025
        "%d-%b-%Y",     # 15-Oct-2025
        "%d/%m/%y"      # 12/03/25
    ]

    parsed_values = []
    error_log = []

    # Loop only through the target column
    for index, raw_val in df[col_name].items():
        if pd.isna(raw_val) or str(raw_val).strip() in ["", "None", "NaN"]:
            parsed_values.append(pd.NaT)
            continue

        clean_val = str(raw_val).strip()
        success = False

        for fmt in expected_formats:
            try:
                parsed_values.append(datetime.strptime(clean_val, fmt).date().isoformat())
                success = True
                break
            except ValueError:
                continue

        if not success:
            # Capture specific row index and offending value
            error_log.append(f"Row {index}: '{raw_val}' cannot be parsed.")
            parsed_values.append(pd.NaT)

    # If any row failed, stop everything and throw a specific error
    if error_log:
        raise ValueError(
            f"Data Integrity Error in column [{col_name}].\n"
            f"Details:\n" + "\n".join(error_log)
        )

    df[col_name] = pd.Series(parsed_values, index=df.index).astype("string")
    return df

In [18]:
def process_date_columns(df, columns_to_parse):
    """
    Checks if specific date columns exist and parses them safely.
    """
    for column in columns_to_parse:
        if column in df.columns:
            try:
                df = parse_date(df, column)
            except Exception as e:
                print(f"Column '{column}' exists, but parsing failed due to data formatting: {e}")
        else:
            print(f"Skipping parsing: '{column}' column is missing from this file.")

    return df

# Pass everything inside a single list
target_dates = ["Invoice Date", "Payment Date", "Session Date", "Start Date"]
df_cleaned = process_date_columns(df_cleaned, target_dates)

Skipping parsing: 'Session Date' column is missing from this file.
Skipping parsing: 'Start Date' column is missing from this file.


In [19]:
# Convert to Decimal and strictly round to 2 decimal places
def to_rounded_decimal(val, places = 2):
    # Catch null and empty strings
    if pd.isna(val) or val in ["nan", ""]:
        return None
    try:
        formats = {
            1: Decimal("0.1"),
            2: Decimal("0.01")
        }

        quantize_format = formats[places]

        return Decimal(str(val)).quantize(quantize_format, rounding=ROUND_HALF_UP)

    except Exception:
        return None

In [20]:
def parse_and_round_decimal(df, column_name, places):
    if column_name not in df.columns:
        print(f"Skipping: Column '{column_name}' does not exist.")
        return df

    try:
        df[column_name] = (
            df[column_name]
            .astype(str)
            .str.extract(r"([\d.,]+)")[0]
            .str.replace(",", "", regex=False)
        )

        df[column_name] = df[column_name].apply(to_rounded_decimal, places = places)

    except Exception as e:
        print(f"Error cleaning '{column_name}': {e}")

    return df

df_cleaned = parse_and_round_decimal(df_cleaned, "Amount", places = 2)
df_cleaned = parse_and_round_decimal(df_cleaned, "Fees Charged", places = 2)
df_cleaned = parse_and_round_decimal(df_cleaned, "Duration (Hours)", places = 1)
df_cleaned = parse_and_round_decimal(df_cleaned, "Hourly Rate (SGD)", places = 2)

Skipping: Column 'Fees Charged' does not exist.
Skipping: Column 'Duration (Hours)' does not exist.
Skipping: Column 'Hourly Rate (SGD)' does not exist.


#### Add styling to header and generate output

In [22]:
try:
    cleaned_data_path = os.path.join(notebook_dir, "data", "cleaned", "lesson_log.xlsx")

    # Defensive check: Automatically create the target folders if they do not exist
    os.makedirs(os.path.dirname(cleaned_data_path), exist_ok=True)

    with pd.ExcelWriter(cleaned_data_path, engine="xlsxwriter") as writer:
        # Drop data into Excel. header=False prevents double headers.
        df_cleaned.to_excel(writer, sheet_name="Reported Data", index=False, startrow=1, header=False)

        workbook = writer.book
        worksheet = writer.sheets["Reported Data"]

        # Define Header Format (Navy Blue background, White Bold text)
        header_format = workbook.add_format({
            "bold": True,
            "bg_color": "#1B3A6B",
            "font_color": "#FFFFFF",
            "align": "left",
            "valign": "vcenter",
            "border": 1
        })

        # Define a format for 1 decimal place columns (like Duration)
        money_format = workbook.add_format({
            "num_format": "#,##0.00",  # Forces 2 decimal places (e.g., 45.00)
        })

        # Define a format for 1 decimal place columns (like Duration)
        duration_format = workbook.add_format({
            "num_format": "0.0",       # Forces 1 decimal place (e.g., 1.5)
        })

        # Write the clean header list onto Row 0 manually
        for col_index, col_name in enumerate(df_cleaned.columns):
            worksheet.write(0, col_index, str(col_name), header_format)

        column_formats = {
            "Amount": money_format,
            "Fees Charged": money_format,
            "Hourly Rate": money_format,
            "Duration (Hours)": duration_format
        }

        for col_index, col_name in enumerate(df_cleaned.columns):
            # Determine the column format, defaulting to None
            column_format = column_formats.get(col_name)

            # Calculate column width
            max_len = max(len(str(col_name)), df_cleaned[col_name].astype(str).str.len().max()) + 5
            final_width = max(10, min(max_len, 50))

            # Apply column settings
            worksheet.set_column(col_index, col_index, final_width, column_format)

    print(f"File completely generated and styled at: {cleaned_data_path}")

except KeyError as e:
    print(f"[DATA ERROR]: Mapping failed due to a missing structural column index: {e}")

except Exception as e:
    print(f"[SYSTEM ERROR]: Failed to generate styled Excel report: {e}")


File completely generated and styled at: /Users/huien/Documents/opus-tuition/src/data_cleaning/data/cleaned/lesson_log.xlsx


#### Validate dataset

In [23]:
# Print the final cleaned results
df_cleaned

,Invoice ID,Tutor ID,Student Name,Invoice Date,Amount,Payment Status,Payment Date,Notes
3,INV-2025-004,TAS-004,Chloe Wong,2025-03-31,75.00,Pending,<NA>,No response from parent
4,INV-2025-005,TAS-006,Ethan Ng,2025-04-30,70.00,Pending,<NA>,<NA>
5,INV-2025-006,TAS-007,Hafiz Bin Aris,2025-06-30,112.50,Pending,<NA>,Note trailing space
9,INV-2025-009,TAS-011,Raeanne Ong,2025-09-30,120.00,Paid,2025-10-04,<NA>
10,INV-2025-003,TAS-003,Muhammad Hafiz,2025-02-28,240.00,Paid,2025-03-03,<NA>
11,INV-2025-010,TAS-012,Jordan Tan,2025-10-31,135.00,Pending,<NA>,<NA>
12,INV-2025-002,TAS-002,Tan Xiu Ying,2025-03-31,90.00,Pending,<NA>,Awaiting bank transfer
13,INV-2025-011,TAS-006,Ethan Ng,2025-05-31,70.00,Paid,2025-06-02,<NA>
14,INV-2025-007,TAS-008,Aisha Binte Yusof,2025-07-31,90.00,Paid,2025-08-02,<NA>
15,INV-2025-012,TAS-009,Tan Xiu Ying,2025-03-31,45.00,Paid,2025-04-01,<NA>


In [24]:
# View summary of dataframe
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 13 entries, 3 to 18
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Invoice ID      13 non-null     string
 1   Tutor ID        13 non-null     string
 2   Student Name    13 non-null     string
 3   Invoice Date    13 non-null     string
 4   Amount          13 non-null     object
 5   Payment Status  13 non-null     string
 6   Payment Date    8 non-null      string
 7   Notes           3 non-null      string
dtypes: object(1), string(7)
memory usage: 936.0+ bytes
